# [1장 1강] - RNN/LSTM의 한계와 Transformer 필요성 (1)

<aside>
🎯

**실습 목표** · 수동 RNN으로 순차 의존을 확인하고, RNN과 Self-Attention의 정보 경로·비용을 수치로 비교한 뒤, 서비스 조건에 맞는 첫 baseline 후보를 근거와 함께 선택합니다.

</aside>

<aside>
🧑‍💻

**진행 방식 및 환경** · Python 3.11/3.12, PyTorch 2.x CPU에서 실행합니다. 각 정답 셀은 독립 실행 가능합니다. 기본 2문제는 필수, 심화 1문제는 선택입니다. 출력값 옆에는 그 값이 구조 선택에 어떤 의미인지 한 문장으로 적으세요.

</aside>

---

## 필수 · 기본 문제 1. Hidden state 전달을 직접 구현하기

### 문제 배경

RNN의 마지막 상태는 마지막 token만의 결과가 아닙니다. 이전 상태가 반복해서 전달되므로 첫 token의 변화가 뒤 상태까지 이어집니다. `nn.RNN` 없이 가장 작은 순환 계산을 구현해 이 계약을 확인합니다.

### 시작 코드

```python
import torch

sequence = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
W_x = torch.tensor([[0.5, -0.2], [0.3, 0.4]])
W_h = torch.tensor([[0.1, 0.2], [-0.3, 0.1]])

def recurrent_encode(sequence, W_x, W_h, h0=None):
    # TODO: 모든 시점의 hidden state를 [L, H]로 반환하세요.
    raise NotImplementedError
```

### 수행 요구사항

1. 초기 상태가 없으면 입력과 같은 dtype의 0 벡터를 만드세요.
2. 각 token에서 `tanh(token @ W_x + hidden @ W_h)`를 계산하세요.
3. 모든 시점 상태를 stack해 반환하세요.
4. 첫 token만 바꾼 입력의 마지막 상태가 원본과 다른지 검증하세요.

### 제출 결과

- 전체 hidden state와 shape
- 원본/변경 입력의 마지막 state
- `기본 문제 1 자동 검증: PASS`
- 순차 의존성을 실행 결과에 근거해 설명한 2문장

### 자동 검증

```python
states = recurrent_encode(sequence, W_x, W_h)
changed = sequence.clone(); changed[0] = torch.tensor([0., 1.])
changed_states = recurrent_encode(changed, W_x, W_h)
assert states.shape == (3, 2)
assert torch.isfinite(states).all()
assert not torch.allclose(states[-1], changed_states[-1])
print("기본 문제 1 자동 검증: PASS")
```

접근 순서 · 먼저 hidden의 초기 shape와 dtype을 입력에 맞춥니다. 반복문에서는 새 hidden을 바로 다음 시점에 넘기고, 모든 시점 값을 list에 모아 마지막에 [L,H]로 stack합니다.

In [1]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)
sequence = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
W_x = torch.tensor([[0.5, -0.2], [0.3, 0.4]])
W_h = torch.tensor([[0.1, 0.2], [-0.3, 0.1]])

def recurrent_encode(sequence, W_x, W_h, h0=None):
    """앞에서부터 token을 읽고 모든 hidden state를 반환합니다."""
    hidden_dim = W_h.size(0)
    # 호출자가 준 h0를 함수 안에서 바꾸지 않도록 clone합니다.
    hidden = (torch.zeros(hidden_dim, dtype=sequence.dtype)
              if h0 is None else h0.clone())
    states = []
    for token in sequence:
        # 이 hidden이 다음 반복에 재사용되는 지점이 시간축 의존성입니다.
        hidden = torch.tanh(token @ W_x + hidden @ W_h)
        states.append(hidden)
    return torch.stack(states)

states = recurrent_encode(sequence, W_x, W_h)
changed = sequence.clone()
changed[0] = torch.tensor([0., 1.])
changed_states = recurrent_encode(changed, W_x, W_h)

print("states:\n", states)
print("original last:", states[-1])
print("changed last:", changed_states[-1])
assert states.shape == (3, 2)
assert torch.isfinite(states).all()
assert not torch.allclose(states[-1], changed_states[-1])
print("기본 문제 1 자동 검증: PASS")

states:
 tensor([[ 0.4621, -0.1974],
        [ 0.3846,  0.4404],
        [ 0.6084,  0.3104]])
original last: tensor([0.6084, 0.3104])
changed last: tensor([0.5938, 0.2806])
기본 문제 1 자동 검증: PASS


## 필수 · 기본 문제 2. 의존 경로와 `L²` 비용 비교하기

### 문제 배경

Self-Attention은 한 층에서 먼 위치를 직접 연결하지만 모든 위치 쌍의 score를 계산합니다. 두 성질을 동시에 숫자로 보고 한쪽 장점만 과장하지 않는 비교표를 만듭니다.

### 시작 코드

```python
sequence_lengths = [4, 16, 128]

def compare_dependency_paths(length):
    # TODO: length, rnn_path, attention_path, attention_score_elements 반환
    raise NotImplementedError
```

### 수행 요구사항

1. `length < 2`이면 `ValueError`를 발생시키세요.
2. RNN 경로는 `length-1`, 한 Self-Attention 층의 경로는 `1`로 계산하세요.
3. 기본 score 원소 수를 `length**2`로 계산하세요.
4. 길이가 32배가 될 때 score 수가 몇 배가 되는지도 설명하세요.

### 제출 결과

- 세 길이의 비교 dictionary
- 짧은 정보 경로의 장점 1문장
- `L²` 메모리 비용의 한계 1문장
- `기본 문제 2 자동 검증: PASS`

### 자동 검증

```python
reports = [compare_dependency_paths(n) for n in sequence_lengths]
assert reports[0]["rnn_path"] == 3
assert reports[-1]["attention_path"] == 1
assert reports[-1]["attention_score_elements"] == 16384
print("기본 문제 2 자동 검증: PASS")
```

    
    **자주 하는 실수**
    
    - RNN 경로를 `length`로 계산해 시작 위치를 한 번 더 셉니다.
    - Attention 경로가 짧다는 사실을 “계산량이 항상 작다”로 해석합니다.
    - `L²`을 전체 모델 파라미터 수로 잘못 부릅니다.

---

풀이 핵심 · 두 구조를 하나의 숫자로 우열 비교하지 않습니다. RNN은 위치 간 최단 경로를, Attention은 한 head의 score 원소 수를 계산해 서로 다른 병목을 나란히 봅니다.

상세 해설 · 첫 상태가 두 번째 계산에, 두 번째가 세 번째 계산에 입력됩니다. 따라서 첫 token을 바꾸면 여러 단계를 거쳐 마지막 상태도 바뀝니다. 이 경로가 순서 정보를 반영하는 장점이면서 긴 문맥에서 직렬 경로가 길어지는 원인입니다.

In [2]:
sequence_lengths = [4, 16, 128]

def compare_dependency_paths(length):
    if length < 2:
        raise ValueError("두 위치를 비교하려면 length는 2 이상이어야 합니다.")
    return {
        "length": length,
        # 첫 위치에서 마지막 위치까지 인접 hidden 전달 횟수입니다.
        "rnn_path": length - 1,
        # 한 Attention 층에서는 두 위치가 직접 score를 가집니다.
        "attention_path": 1,
        # Head와 batch를 생략한 기본 L x L score 수입니다.
        "attention_score_elements": length ** 2,
    }

reports = [compare_dependency_paths(n) for n in sequence_lengths]
for report in reports:
    print(report)

assert reports[0]["rnn_path"] == 3
assert reports[-1]["attention_path"] == 1
assert reports[-1]["attention_score_elements"] == 16384
try:
    compare_dependency_paths(1)
    raise AssertionError("길이 검증이 동작하지 않았습니다.")
except ValueError:
    pass
print("기본 문제 2 자동 검증: PASS")

{'length': 4, 'rnn_path': 3, 'attention_path': 1, 'attention_score_elements': 16}
{'length': 16, 'rnn_path': 15, 'attention_path': 1, 'attention_score_elements': 256}
{'length': 128, 'rnn_path': 127, 'attention_path': 1, 'attention_score_elements': 16384}
기본 문제 2 자동 검증: PASS


상세 해설 · 길이가 4에서 128로 32배가 되면 score 수는 16에서 16,384로 1,024배가 됩니다. 실제 메모리는 batch·head·dtype·중간 activation도 포함하므로 이 계산은 하한에 가까운 구조 비교입니다.